# Итоговый отчёт по модели SIR

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт загружает ранее сохранённые результаты моделирования SIR
и строит сравнительные графики для итогового отчёта.

### Что делает скрипт

1. Загружает результаты трёх предыдущих экспериментов:
   - Детерминированная симуляция (`sir_det.csv`)
   - Стохастическая симуляция (`sir_stoch.csv`)
   - Сканирование параметра β (`sir_scan.csv`)

2. Строит два итоговых графика:
   - Сравнение детерминированной и стохастической динамики I(t)
   - Зависимость пика инфицированных от коэффициента заражения β

## Входные данные

| Файл | Создан скриптом | Описание |
|------|-----------------|----------|
| `data/sir_det.csv` | Базовый прогон | Детерминированная симуляция |
| `data/sir_stoch.csv` | Базовый прогон | Стохастическая симуляция |
| `data/sir_scan.csv` | Сканирование β | Результаты параметрического анализа |

**Важно:** Эти файлы должны быть созданы до запуска данного скрипта.

## Выходные данные

| Файл | Описание | Рисунок |
|------|----------|---------|
| `plots/comparison.png` | Сравнение I(t) детерминированной и стохастической симуляций | Рис. 6.5 |
| `plots/sensitivity.png` | Зависимость пика I от коэффициента β | Рис. 6.6 |

## Интерпретация результатов

### Рисунок 6.5: Сравнение детерминированной и стохастической динамики

- При большом размере популяции (N = 1000) стохастическая траектория
  близка к детерминированной

- Стохастическая кривая может иметь небольшие флуктуации

- Различия между кривыми демонстрируют влияние случайности
  на динамику эпидемии

### Рисунок 6.6: Зависимость пика I от β (чувствительность)

- Позволяет количественно оценить, как изменение заразности β
  влияет на тяжесть эпидемии

- Важно для принятия решений (например, оценка эффекта от мер
  по снижению β: карантин, маски, социальная дистанция)

- Демонстрирует нелинейный характер зависимости: малые изменения β
  в критической области приводят к значительным изменениям пика I

## Инициализация проекта DrWatson

In [1]:
using DrWatson
@quickactivate "project"

## Подключение утилит для работы с данными и графикой

In [2]:
using DataFrames, CSV, Plots

## Загрузка результатов экспериментов

### Загрузка детерминированной симуляции

Файл содержит колонки: time, S, I, R
Параметры симуляции: β = 0.3, γ = 0.1, tmax = 100.0

In [3]:
df_det = CSV.read(datadir("sir_det.csv"), DataFrame)

Row,time,S,I,R
,Float64,Float64,Float64,Float64
1,0.0,990.0,10.0,0.0
2,0.5,2.47053e-7,952.691,47.3086
3,1.0,3.28252e-7,906.228,93.7719
4,1.5,-1.78365e-7,862.031,137.969
5,2.0,-2.6597e-7,819.989,180.011
6,2.5,2.389e-7,779.998,220.002
7,3.0,-3.23518e-7,741.957,258.043
8,3.5,-2.30682e-7,705.771,294.229
9,4.0,3.04028e-8,671.35,328.65


### Загрузка стохастической симуляции

Файл содержит колонки: time, S, I, R
Параметры симуляции: β = 0.3, γ = 0.1, tmax = 100.0
Зерно ГСЧ: 123 (воспроизводимость)

In [4]:
df_stoch = CSV.read(datadir("sir_stoch.csv"), DataFrame)

Row,time,S,I,R
,Float64,Float64,Float64,Float64
1,0.0,990.0,10.0,0.0
2,0.000219318,989.0,11.0,0.0
3,0.00025471,988.0,12.0,0.0
4,0.000435457,987.0,13.0,0.0
5,0.00124186,986.0,14.0,0.0
6,0.00137312,985.0,15.0,0.0
7,0.00151759,984.0,16.0,0.0
8,0.00219412,983.0,17.0,0.0
9,0.00239638,982.0,18.0,0.0


### Загрузка результатов сканирования β

Файл содержит колонки: β, peak_I, final_R
Диапазон β: 0.1 : 0.05 : 0.8

In [5]:
df_scan = CSV.read(datadir("sir_scan.csv"), DataFrame)

Row,β,peak_I,final_R
,Float64,Float64,Float64
1,0.1,10.0,82.8346
2,0.15,69.7245,468.15
3,0.2,158.441,775.694
4,0.25,237.513,888.459
5,0.3,303.791,939.338
6,0.35,359.197,965.549
7,0.4,405.889,979.935
8,0.45,445.723,988.122
9,0.5,479.99,992.882


## Построение итоговых графиков

### Рисунок 6.5: Сравнение детерминированной и стохастической динамики

На одном графике отображаются две кривые I(t):
- **Deterministic I** (синяя линия) — усреднённая динамика из ОДУ
- **Stochastic I** (красная линия) — траектория из алгоритма Гиллеспи

Особенности:
- Стохастическая кривая обрезается до длины детерминированной
  для корректного сравнения
- Оси: время по горизонтали, число инфицированных по вертикали

In [6]:
p1 = plot(
    df_det.time,
    [df_det.I df_stoch.I[1:length(df_det.time)]],
    label = ["Deterministic I" "Stochastic I"],
    xlabel = "Time",
    ylabel = "Infected",
    title = "Comparison",
)

savefig(plotsdir("comparison.png"))

Could not create decoration from factory! Running with no decorations.


"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/plots/comparison.png"

### Рисунок 6.6: Зависимость пика I от β (чувствительность)

График отображает зависимость максимального числа инфицированных
от коэффициента заражения β.

Характерные особенности:
- Пороговый эффект: при малых β эпидемия не развивается (peak I ≈ 0)
- Резкий нелинейный рост в критической области
- Насыщение при больших β (почти всё население переболевает)

In [7]:
p2 = plot(
    df_scan.β,
    df_scan.peak_I,
    marker = :circle,
    xlabel = "β",
    ylabel = "Peak I",
    title = "Sensitivity",
)

savefig(plotsdir("sensitivity.png"))

Could not create decoration from factory! Running with no decorations.


"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/plots/sensitivity.png"

## Завершение работы

Скрипт успешно выполнил:
- Загрузка трёх CSV-файлов с результатами
- Построение графика сравнения → `plots/comparison.png`
- Построение графика чувствительности → `plots/sensitivity.png`

In [8]:
println("Отчётные графики сохранены в plots/")

Отчётные графики сохранены в plots/
